In [ ]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset
from datasets import load_dataset

from transformers import (
    set_seed,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

import torch
def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
from huggingface_hub import hf_hub_download
import lightgbm as lgb

params = {
    'objective': 'binary',  
    'metric': 'binary_logloss', 
    'boosting_type': 'gbdt',  
    'num_leaves': 34, 
    'learning_rate': 0.05,  
    'feature_fraction': 0.9, 
    'bagging_fraction': 0.8, 
    'bagging_freq': 5,  
    'verbose': -1, 
}

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
# dataset = load_dataset("")
# human_df  = pd.DataFrame(dataset['main'])

human_df = pd.read_csv('7_human_dataset.csv')
human_df['ds'].value_counts()

ds
ViHSD           30728
HateXplain      20145
Sexism          13631
GermEval2019    12536
GermEval2021     4188
US_election      2400
Covid            2290
Name: count, dtype: int64

# 2 Labels - Lgb Models

In [7]:
df_1 = human_df[human_df['ds'] == 'ViHSD'].sample(10000, random_state=42)
df_2 = human_df[human_df['ds'] != 'ViHSD']
lgb_df = pd.concat([df_1, df_2]).reset_index()
lgb_df['ds'].value_counts()

ds
HateXplain      20145
Sexism          13631
GermEval2019    12536
ViHSD           10000
GermEval2021     4188
US_election      2400
Covid            2290
Name: count, dtype: int64

Load lightgbm data, with labeled prob.

In [ ]:
dataset = load_dataset("anonymousOWSHateLLM/lgb_data_2_label")
lgb_df  = pd.DataFrame(dataset['lgb_data_2_label'])
# lgb_df = pd.read_csv('lgb_data_2_label.csv')
# lgb_df['ds'].value_counts()

ds
HateXplain      20022
Sexism          13631
GermEval2019    12131
ViHSD           10000
GermEval2021     3457
Covid            2164
US_election      2107
Name: count, dtype: int64

In [5]:
mstral7b = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
gemma9b = "unsloth/gemma-2-9b-it-bnb-4bit"
qwen14b = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
llama8B = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
columns = [qwen14b, llama8B, mstral7b, gemma9b]

In [6]:
lgb_df['label_id'].value_counts()

label_id
2    41888
1    21624
Name: count, dtype: int64

In [ ]:
lgb_df = lgb_df.sample(frac=1, random_state=def_seed)

In [7]:
c1 = [x + "_label_1" for x in columns]
X = lgb_df[c1]
y = lgb_df['label_id'] == 1
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)



model_label_1 = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=1000,
)

val_preds = model_label_1.predict(X_val, num_iteration=model_label_1.best_iteration)
print(roc_auc_score(y_val, val_preds))


0.8544857422952564


In [8]:
c2 = [x + "_label_2" for x in columns]
X = lgb_df[c2]
y = lgb_df['label_id'] == 2
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)


model_label_2 = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=1000,
)

val_preds = model_label_2.predict(X_val, num_iteration=model_label_2.best_iteration)
print(roc_auc_score(y_val, val_preds))

0.8557401191787175


# 3 Labels:

Create data for training LightGbm models for 3 labels task: Combination of HateOff and HateXplain

Load HateOff data:


@inproceedings{hateoffensive,
  title = {Automated Hate Speech Detection and the Problem of Offensive Language},
  author = {Davidson, Thomas and Warmsley, Dana and Macy, Michael and Weber, Ingmar}, 
  booktitle = {Proceedings of the 11th International AAAI Conference on Web and Social Media},
  series = {ICWSM '17},
  year = {2017},
  location = {Montreal, Canada},
  pages = {512-515}
  }


In [ ]:
df_hate_off = pd.read_csv("df_hate_off.csv")
df_hate_off = df_hate_off[['tweet', 'class']]
df_hate_off['multi_label_id'] = df_hate_off['class'] + 1
df_hate_off['text'] = df_hate_off['tweet']
df_hate_off.multi_label_id.value_counts()

multi_label_id
2    19190
3     4163
1     1430
Name: count, dtype: int64

In [10]:
hateXplain = human_df[human_df['ds'] == 'HateXplain']
Lgb_df_3_labels = pd.concat([df_hate_off, hateXplain]).reset_index()
Lgb_df_3_labels.multi_label_id.value_counts()

multi_label_id
2    24669
3    11977
1     8282
Name: count, dtype: int64

In [11]:
Lgb_df_3_labels.shape

(44928, 8)

Load labeled dataset, contain probs of 3 models: Mistral-7B, Gemma2-9B and Qwen2.5-14B

In [ ]:
dataset = load_dataset("anonymousOWSHateLLM/lgb_data_3_label")
Lgb_df_3_labels  = pd.DataFrame(dataset['lgb_data_3_label'])
# Lgb_df_3_labels = pd.read_csv('lgb_data_3_label.csv')
# Lgb_df_3_labels.label_id.value_counts()

label_id
2    24611
3    11936
1     8228
Name: count, dtype: int64

In [ ]:
Lgb_df_3_labels = Lgb_df_3_labels.sample(frac=1, random_state=def_seed)


In [13]:
columns =  [mstral7b, gemma9b, qwen14b]
for col in columns:
    probabilities = np.array([Lgb_df_3_labels[col +"_label_1"], Lgb_df_3_labels[col +"_label_2"], Lgb_df_3_labels[col +"_label_3"]]).T  
    y_pred = np.argmax(probabilities, axis=1)
    y_true = np.array(Lgb_df_3_labels["label_id"])
    print(f1_score(y_true - 1, y_pred, average="macro"))


0.5280181109851637
0.5790325650700758
0.6132282873259557


In [14]:
all_cols = []
for col in columns:
    all_cols.append(col + "_label_1")
    all_cols.append(col + "_label_2")
    all_cols.append(col + "_label_3")


X = Lgb_df_3_labels[all_cols]
y = Lgb_df_3_labels['label_id']  - 1

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

params = {
    'objective': 'multiclass', 
    'num_class': len(y.unique()),
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt', 
    'num_leaves': 34,  
    'learning_rate': 0.05, 
    'feature_fraction': 0.9, 
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,  
}

model = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=1000,

)

val_preds = model.predict(X_val, num_iteration=model.best_iteration)
val_preds = [list(x).index(max(x)) for x in val_preds]


In [15]:
f1_score = f1_score(y_val, val_preds, average="macro")
print(f1_score)

0.6848470652852899
